### 注：内置中间件就是官方写的中间件，可以直接调用，在langchain.agents.middleware库中

In [2]:
#调用模型
import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 定义LLM模型
model = ChatOpenAI(model="gpt-4o-mini", temperature=0, timeout=10,max_tokens=1000)

### 1、综述
##### 在接近token限制时自动汇总对话历史记录。相当于v0.3某个对记忆的处理函数，忘了叫啥了

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="gpt-4o",
    tools=[...],
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            max_tokens_before_summary=4000,  # Trigger summarization at 4000 tokens
            messages_to_keep=20,  # Keep last 20 messages after summary
            summary_prompt="Custom prompt for summarization...",  # Optional
        ),
    ],
)

### 2、人机协同中间件（个人感觉挺重要的）
##### 它会在 Agent 真正调用工具前，先把控制权交给你，让你决定“这条操作能不能继续、要不要改、还是直接拒绝”。具体行为由 interrupt_on 参数控制

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.tools import tool

@tool
def send_email_tool():
    ...

@tool
def read_email_tool():
    ...

agent = create_agent(
    model="gpt-4o",
    tools=[send_email_tool, read_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                # Require approval, editing, or rejection for sending emails
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                # Auto-approve reading emails
                "read_email_tool": False,
            }
        ),
    ],
)

##### PS:interrupt_on中的参数
##### ①对 send_email_tool:
##### 只要 Agent 想发邮件，就会弹出一个中断，界面上会出现“approve / edit / reject”三个选项。你必须点“approve”它才会真正发；点“edit”可以先改收件人、标题、正文再发；点“reject”就跳过这次调用。
##### 目的：防止 AI 误发或滥发邮件，造成不可撤回的后果。
##### ②对 read_email_tool
##### 设为 False，表示不中断，Agent 读邮件时可直接执行，无需人工确认。
##### 目的：只把“写”操作（发邮件）纳入人工审核，“读”操作保持流畅。

### 3、Anthropic prompt缓存
##### 通过使用 Anthropic 模型缓存重复的提示前缀来降低成本。
##### Anthropic 是目前唯一一家把「cache_control」字段写进公开 API 的大厂。这个对本地大模型没用，本质上还是要在Anthropic服务器上做 tokenize → 算 KV-Cache → 存到自家分布式显存 → 正常推理。之后同样 hash 的请求：直接拿上次存好的 KV-Cache 接着推理，跳过重复 tokenize 和预填，账单按折扣价计。“省”的是 Anthropic 服务器侧重复计算 KV-Cache 的步骤，用Anthropic 模型能节省一定成本

In [ ]:
from langchain.messages import HumanMessage
from langchain_anthropic import ChatAnthropic
from langchain_anthropic.middleware import AnthropicPromptCachingMiddleware
from langchain.agents import create_agent


LONG_PROMPT = """
Please be a helpful assistant.

<Lots more context ...>
"""

agent = create_agent(
    model=ChatAnthropic(model),
    system_prompt=LONG_PROMPT,
    middleware=[AnthropicPromptCachingMiddleware(ttl="5m")],
)

# cache store
agent.invoke({"messages": [HumanMessage("Hi, my name is Bob")]})

# cache hit, system prompt is cached
agent.invoke({"messages": [HumanMessage("What's my name?")]})

### 4、模型调用限制
##### 限制模型调用次数，防止无限循环或成本过高。

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware


agent = create_agent(
    model="gpt-4o",
    tools=[...],
    middleware=[
        ModelCallLimitMiddleware(
            thread_limit=10,  # Max 10 calls per thread (across runs)
            run_limit=5,  # Max 5 calls per run (single invocation)
            exit_behavior="end",  # Or "error" to raise exception
        ),
    ],
)

### 5、工具调用限制
##### 通过限制工具调用的数量来控制代理执行，无论是跨所有工具的全局调用还是针对特定工具。
##### 可全局限制跨所有工具；要特定工具的工具调用，请设置tool_name 。对于每个限制，请指定以下一项或两项：
##### ①线程限制 （） - 对话中所有运行的最大调用数。跨调用持续存在。需要检查指针。thread_limit
##### ②运行限制 （） - 每次调用的最大调用次数。每回合重置一次。run_limit

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware
from langchain.tools import tool

@tool
def search_tool():
    ...

@tool
def database_tool():
    ...

@tool
def scraper_tool():
    ...

# Global limit: max 20 calls per thread, 10 per run
global_limiter = ToolCallLimitMiddleware(
    thread_limit=20,
    run_limit=10,
)

# Tool-specific limit with default "continue" behavior
search_limiter = ToolCallLimitMiddleware(
    tool_name="search",
    thread_limit=5,
    run_limit=3,
)

# Thread limit only (no per-run limit)
database_limiter = ToolCallLimitMiddleware(
    tool_name="query_database",
    thread_limit=10,
)

# Strict enforcement with "error" behavior
web_scraper_limiter = ToolCallLimitMiddleware(
    tool_name="scrape_webpage",
    run_limit=2,
    exit_behavior="error",
)

# Immediate termination with "end" behavior
critical_tool_limiter = ToolCallLimitMiddleware(
    tool_name="delete_records",
    run_limit=1,
    exit_behavior="end",
)

# Use multiple limiters together
agent = create_agent(
    model="gpt-4o",
    tools=[search_tool, database_tool, scraper_tool],
    middleware=[
        global_limiter,
        search_limiter,
        database_limiter,
        web_scraper_limiter
    ],
)

### 6、模型回退
##### 当主模型失败时，自动回退到替代模型。

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelFallbackMiddleware


agent = create_agent(
    model="gpt-4o",  # Primary model
    tools=[...],
    middleware=[
        ModelFallbackMiddleware(
            "gpt-4o-mini",  # Try first on error
            "claude-3-5-sonnet-20241022",  # Then this
        ),
    ],
)

### 7、PII 检测（个人感觉比较重要）
##### 检测和处理对话中的个人身份信息。
##### 就是如果检测到传入的隐私信息，则对隐私信息做处理，保护个人信息。

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware


agent = create_agent(
    model="gpt-4o",
    tools=[...],
    middleware=[
        # Redact emails in user input
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        # Mask credit cards (show last 4 digits)
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
        # Custom PII type with regex
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",  # Raise error if detected
        ),
    ],
)

##### （1）. 邮箱地址处理：PIIMiddleware("email", strategy="redact", apply_to_input=True)
##### 检测内容：用户输入中是否包含邮箱地址（如 user@example.com）。
##### 处理方式：完全 redact（打码），例如变成 [EMAIL] 或 ***@***.com。
##### 目的：防止邮箱泄露。
##### （2）. 信用卡号处理：PIIMiddleware("credit_card", strategy="mask", apply_to_input=True)
##### 检测内容：信用卡号（如 4111111111111111）。
##### 处理方式：mask（掩码），只保留最后 4 位，如 ****-****-****-1111。
##### 目的：保留部分信息用于识别，但隐藏大部分敏感数据。
#####  （3）、API Key 检测：PIIMiddleware("api_key",detector=r"sk-[a-zA-Z0-9]{32}",strategy="block",)
##### 检测内容：符合正则表达式 sk-[a-zA-Z0-9]{32} 的字符串（如 OpenAI 的 API Key）。
##### 处理方式：block（阻断），一旦检测到，直接抛出错误，不让请求继续。
##### 目的：防止用户无意中把 API Key 发给模型，造成泄露。

### 8、待办事项清单（Todolist）
##### 为座席提供复杂的多步骤任务的任务规划和跟踪功能。
##### 他与日志的区别在于：日志是“被动流水账”，TodoList 是“让大模型主动写任务看板并实时打勾”。也就是说日志就是记录了每一步操作，但是TodoLis是把问题抛给大模型，让他来制定每一个步骤，然后制作成类似于表的结构，然后可以更新它每一步的状态（比如已完成、待办、未开始等）

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import TodoListMiddleware
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool


@tool
def read_file(file_path: str) -> str:
    """Read contents of a file."""
    with open(file_path) as f:
        return f.read()


@tool
def write_file(file_path: str, content: str) -> str:
    """Write content to a file."""
    with open(file_path, 'w') as f:
        f.write(content)
    return f"Wrote {len(content)} characters to {file_path}"


@tool
def run_tests(test_path: str) -> str:
    """Run tests and return results."""
    # Simplified for example
    return "All tests passed!"


agent = create_agent(
    model="gpt-4o",
    tools=[read_file, write_file, run_tests],
    middleware=[TodoListMiddleware()],
)

result = agent.invoke({
    "messages": [HumanMessage("Refactor the authentication module to use async/await and ensure all tests pass")]
})

# The agent will use write_todos to plan and track:
# 1. Read current authentication module code
# 2. Identify functions that need async conversion
# 3. Refactor functions to async/await
# 4. Update function calls throughout codebase
# 5. Run tests and fix any failures

print(result["todos"])  # Track the agent's progress through each step

### 9、LLM 工具选择器
##### 在调用主模型之前，使用 LLM 智能地选择相关工具。

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import LLMToolSelectorMiddleware


agent = create_agent(
    model="gpt-4o",
    tools=[...],  # Many tools
    middleware=[
        LLMToolSelectorMiddleware(
            model="gpt-4o-mini",  # Use cheaper model for selection
            max_tools=3,  # Limit to 3 most relevant tools
            always_include=["search"],  # Always include certain tools
        ),
    ],
)

##### 工作流程（对每次用户输入）：
##### （1）小模型（gpt-4o-mini）把用户问题 + 全部工具描述过一遍，打分排序。
##### （2）按 max_tools 截断，再补上 always_include 列表。
##### （3）最终只有 “精选子集” 被塞进主模型的 System Prompt，Agent 接着推理、调用。
##### 带来的好处：
##### 省 token 费——大模型只看见 3 个工具，不是 30 个。
##### 减少误判——工具越少，LLM 越不“眼花”，选对工具的概率提高。
##### 速度更快——短 Prompt →  faster generation。
##### 灵活白名单——always_include 保证关键工具（如搜索）不被误筛掉。

### 10、工具重试
##### 使用可配置的指数退避自动重试失败的工具调用。
##### 这个就是中间件先调用tool，尝试tool是否运行正常，如果失败，则调用多次（自己定），直至成功或者超过定义的最大时限。

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolRetryMiddleware


agent = create_agent(
    model,
    tools=[search_tool, database_tool],
    middleware=[
        ToolRetryMiddleware(
            max_retries=3,  # 如果调用工具失败，则再尝试3次，如果都不成功再抛异常
            backoff_factor=2.0,  # “每次等待时间” = 上一次等待时间 × backoff_factor，减少服务器负荷
            initial_delay=1.0,  # 设定第一次调用失败后的等待时间，单位为秒
            max_delay=60.0,  # 总共等待时长为60秒
            jitter=True,  # 因为调用工具失败可能是多个工具同时失败，于是随机分散每个调用工具的请求时间，防止多个请求同时发送给服务器导致服务器负荷大
        ),
    ],
)

### 11、LLM 工具模拟器
##### 使用LLM模拟工具执行以进行测试，用人工智能生成的响应替换实际的工具调用。
#### 有几个要点说一下，1.这个中间件只是一个“开发脚手架”，上线前必须拆掉，正常运行时根本不该让它出现；2.LLMToolEmulator本质上是通过调用model编一个假结果验证agent能否调用工具和正常传递参数以及工具描述，实质上并没有调用工具，也没有通过工具调用到真实数据；3.它只能验证“Agent → 工具”这一层决策链

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import LLMToolEmulator
from langchain.tools import tool


@tool
def get_weather():
    ...

@tool
def search_database():
    ...

@tool
def send_email():
    ...

agent = create_agent(
    model="gpt-4o",
    tools=[get_weather, search_database, send_email],
    middleware=[
        # 模拟所有工具
        LLMToolEmulator(),

        # 只模拟指定工具
        # LLMToolEmulator(tools=["get_weather", "search_database"]),

        # 用指定模型进行模拟
        # LLMToolEmulator(model="claude-sonnet-4-5-20250929"),
    ],
)

### 12、对工具消息进行上下文编辑
##### 通过修剪、总结或清除工具使用来管理对话上下文。
#### 注：只清理有关工具的消息，比如ToolMessages

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ContextEditingMiddleware, ClearToolUsesEdit


agent = create_agent(
    model="gpt-4o",
    tools=[...],
    middleware=[
        ContextEditingMiddleware(
            edits=[
                ClearToolUsesEdit(trigger=1000),  # 当有关tool的token数到达1000时，则会删除历史中所有的工具调用记录
            ],
        ),
    ],
)